# PyRosetta 核心概念

对应笔记仓库 `037_Rosetta_Notes/Rosetta_Concepts.md`

## 01  Pose

Pose 是 Rosetta 表示「一个分子体系当前状态」的中心对象，里面装着：构象（xyz + 二面角）、序列与化学、链拓扑、上次打分的能量、以及 PDBInfo（原始 PDB 的链号与残基编号）。

两个要点：

1. **Pose 是可变的** —— 所有 Mover 都是原地修改传进去的 Pose，不返回新对象。
2. **两套残基编号** —— Rosetta 内部从 1 连续编到 N、无视链边界；PDB 文件按链分别计数。两者靠 PDBInfo 换算。

In [1]:
import pyrosetta

pyrosetta.init('-mute all')    # 每个进程只需 init 一次，加载数据库要十几秒

┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │
│               Created in JHU by Sergey Lyskov and PyRosetta Team              │
│               (C) Copyright Rosetta Commons Member Institutions               │
│                                                                               │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRES PURCHASE OF A LICENSE │
│          See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└───────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2026 [Rosetta PyRosetta4.Release.python311.ubuntu 2026.29+releasequarterly.80a0635615099e1b918474a63acba7b1de6fd107 2026-07-14T16:24:11] retrieved from: http://www.pyrosetta.org


In [2]:
pose = pyrosetta.pose_from_sequence('AAAGGGKKK')    # 不用文件，直接从序列造一个 Pose

print(pose.total_residue())          # 残基总数
print(pose.sequence())               # 序列
print(pose.residue(3).name3())       # 第 3 个残基的三字母名（注意从 1 开始数）
print(pose.phi(3), pose.psi(3))      # 第 3 个残基的主链二面角

9
AAAGGGKKK
ALA
180.0 180.0


In [3]:
print(pose)    # Pose 的摘要：序列、折叠树、链信息

PDB file name: AAAGGGKK
Total residues: 9
Sequence: AAAGGGKKK
Fold tree:
FOLD_TREE  EDGE 1 9 -1 


### 01-2  加载真实复合物

`8hpu_M_N_A.pdb` 是一个抗体-抗原复合物：H 链（VH 121 aa）、L 链（VL 109 aa）、A 链（抗原 194 aa），共 424 残基。
只有 ATOM 记录，无水、无配体、无 altloc。

In [4]:
pdb = '/data/lmk/rosetta_inputs/8hpu_M_N_A.pdb'
pose = pyrosetta.pose_from_pdb(pdb)

print('总残基数:', pose.total_residue())
print('链数:', pose.num_chains())
print('前 30 个残基:', pose.sequence()[:30])

总残基数: 424
链数: 3
前 30 个残基: VQLVESGGGLVQPGGSLRLSCAASEITVSS


**两套编号的对照** —— PDBInfo 是连接 Rosetta 内部编号和原始 PDB 编号的桥梁。

In [5]:
info = pose.pdb_info()

for ch in range(1, pose.num_chains() + 1):
    b, e = pose.chain_begin(ch), pose.chain_end(ch)    # 该链在 Rosetta 编号里的起止
    print(f'链 {info.chain(b)}   Rosetta {b:>4} - {e:<4}   PDB {info.number(b):>4} - {info.number(e):<4}')

链 H   Rosetta    1 - 121    PDB    1 - 121 
链 L   Rosetta  122 - 230    PDB    1 - 109 
链 A   Rosetta  231 - 424    PDB    1 - 194 


可以看到每条链的 PDB 编号都从头开始，而 Rosetta 编号一路连续往下数。

下面是双向换算，**以后选残基必用**。

In [6]:
print('L 链第 30 位  ->  Rosetta 编号', info.pdb2pose('L', 30))    # PDB -> Rosetta
print('Rosetta 150   ->  PDB', info.pose2pdb(150))               # Rosetta -> PDB

i = info.pdb2pose('L', 30)
print('核对:', pose.residue(i).name3(), '在', info.pose2pdb(i))

L 链第 30 位  ->  Rosetta 编号 151
Rosetta 150   ->  PDB 29 L 
核对: SER 在 30 L 


In [7]:
chains = pose.split_by_chain()    # 按链拆成独立的 Pose

for k in range(1, len(chains) + 1):
    print(k, chains[k].total_residue(), chains[k].sequence()[:20])

1 121 VQLVESGGGLVQPGGSLRLS
2 109 DIQMTQSPSSLSASVGDRVS
3 194 NLCPFDEVFNATRFASVYAW


## 02  ScoreFunction

打分函数：输入一个 Pose，输出一个数（REU）。所有判断——构象好不好、突变有没有改善、binder 值不值得做——最终都归结为比较这个数。

总分 = Σ（权重 × 能量项）。REF2015 共十几项，半物理半统计：一部分来自物理公式（范德华、静电），一部分来自 PDB 数据库统计（rotamer 频率、Ramachandran 分布）。

| 类别 | 能量项 | 含义 |
| :--- | :--- | :--- |
| 范德华 | `fa_atr` / `fa_rep` | 原子间吸引 / 排斥（碰撞惩罚） |
| 溶剂化 | `fa_sol` `lk_ball_wtd` | 把极性基团埋进疏水环境的代价 |
| 静电 | `fa_elec` | 带电与极性基团间的库仑作用 |
| 氢键 | `hbond_sr_bb` `hbond_lr_bb` `hbond_bb_sc` `hbond_sc` | 按主链 / 侧链组合分四类 |
| 构象统计 | `fa_dun` | 侧链构象在 rotamer 库里常不常见 |
|  | `rama_prepro` `p_aa_pp` `omega` | 主链二面角是否落在允许区 |
| 参考态 | `ref` | 每种氨基酸的基线能量，做设计时防止过度偏好 |

两个性质：

1. **单位是 REU**，与 kcal/mol 数量级接近但未经标定 —— 不要写成 kcal/mol，也不要换算 Kd。
2. **是广延量**，蛋白越大总分越负，所以总分不能跨体系比较。评估 binder 要用分离前后的差值，而不是总分。

In [8]:
scorefxn = pyrosetta.get_fa_scorefxn()    # 默认就是 REF2015

total = scorefxn(pose)
print('总分:', total, 'REU')

总分: 313.25165813278215 REU


**各能量项的 REU 分解** —— 从 Python 侧读 `pose.energies()` 再乘以权重。

In [9]:
import pandas as pd

scorefxn(pose)                          # 必须先打分，energies 才有效
e = pose.energies().total_energies()    # 各能量项的未加权总和

rows = []
for st in scorefxn.get_nonzero_weighted_scoretypes():
    raw = e[st]
    w = scorefxn.get_weight(st)
    rows.append({'能量项': str(st).replace('ScoreType.', ''),
                 '原始值': round(raw, 2),
                 '权重': w,
                 '加权贡献': round(raw * w, 2)})

df = pd.DataFrame(rows).sort_values('加权贡献', ascending=False, ignore_index=True)

print('加权合计:', round(df['加权贡献'].sum(), 2),
      '  |  scorefxn(pose):', round(scorefxn(pose), 2))    # 自检：两者应当相等
df

加权合计: 313.25   |  scorefxn(pose): 313.25


,能量项,原始值,权重,加权贡献
0,fa_sol,1316.69,1.000,1316.69
1,fa_dun,1347.38,0.700,943.17
2,fa_rep,1215.08,0.550,668.30
3,ref,178.10,1.000,178.10
4,fa_intra_sol_xover4,84.25,1.000,84.25
5,pro_close,54.58,1.250,68.22
6,rama_prepro,143.09,0.450,64.39
7,omega,149.10,0.400,59.64
8,fa_intra_rep,972.70,0.005,4.86
9,yhh_planarity,0.00,0.625,0.00
